# Literature review with citeformer

This notebook walks through a realistic academic-adjacent workflow:

1. **Research question** — "What are the main contributions of prompt-based reasoning techniques in large language models?"
2. **Fetch** — grab six relevant arXiv papers as `Source` objects. `abstract → content`, `CSL-JSON → metadata`.
3. **Generate** — let a small instruction-tuned model write a paragraph-length review under `Policy.REQUIRED`. Every sentence ends with a `[N]` marker. **The grammar layer makes `[7]` or `[99]` structurally impossible to sample.**
4. **Verify** — run NLI over every emitted citation. Does each source actually entail the sentence that cited it?
5. **Render** — produce APA-7 bibliography entries. The model never touches the bibliography; the formatter does.
6. **Compare** — same prompt, no grammar. Show the contrast.

The point of this notebook isn't to produce a publishable review — small models at 180 tokens can't write one. The point is to show every link in the chain (fetch → sources → grammar-constrained generation → NLI verify → deterministic render) working end-to-end, with the structural guarantee holding on every emitted citation.

## Setup

In [ ]:
# Requires:  pip install 'citeformer[hf,verify]'
import os
import sys

from citeformer import Citeformer, Policy, Source, build_rag_prompt, deduplicate_adjacent_cites
from citeformer.backends.hf import HFBackend

## 1. Fetch six papers from arXiv

We're using the reasoning / prompt-engineering corner of the literature. `Source.from_arxiv` takes the arXiv id, calls the API, and returns a `Source` with `content = <abstract>` and `metadata = <CSL-JSON>`. Responses are disk-cached (`~/.cache/citeformer/metadata/`) so re-running doesn't hammer arXiv.

In [ ]:
ARXIV_IDS = [
    "2201.11903",  # Chain-of-Thought prompting (Wei et al.)
    "2203.11171",  # Self-consistency (Wang et al.)
    "2205.11916",  # Let's Think Step by Step (Kojima et al.)
    "2305.10601",  # Tree of Thoughts (Yao et al.)
    "2304.09842",  # ReAct (Yao et al.)
    "2306.04528",  # Plan-and-Solve (Wang et al.)
]

sources = [Source.from_arxiv(aid) for aid in ARXIV_IDS]
for i, s in enumerate(sources, start=1):
    title = s.metadata.get("title", "(no title)")[:72]
    year = s.metadata.get("issued", {}).get("date-parts", [[None]])[0][0]
    print(f"[{i}] {title} ({year})")

## 2. Build the RAG prompt

`build_rag_prompt` lays out the canonical citeformer prompt: system instruction, a tiny few-shot example, the numbered sources with abstracts, then the user query. Every source in the list becomes a `[i]` marker — and the grammar layer will later mask anything but `[1]..[6]` at decode time.

In [ ]:
prompt = build_rag_prompt(
    query=(
        "Write four citation-dense sentences describing the main contributions "
        "of prompt-based reasoning techniques in large language models. "
        "Cite at least one source in every sentence."
    ),
    sources=sources,
    system=(
        "You are writing a brief technical literature review. "
        "Cite every claim using the provided numbered sources."
    ),
    example=(
        "Chain-of-thought prompting elicits step-by-step reasoning in LLMs [1]. "
        "Self-consistency decoding then ensembles multiple reasoning paths [2]."
    ),
    answer_prefix="Review:",
)
print(prompt[:600] + "\n...")

## 3. Generate the review — grammar-constrained

We use a small model (Qwen 2.5 0.5B Instruct, ~500 MB) so the notebook runs on a commodity laptop. Any instruction-tuned model works here — pass the HF id to `HFBackend(model=...)`.

Under `Policy.REQUIRED`, every sentence must end in a `[N] [M]` group. The digit enum is bounded by `range(1, len(sources)+1)` at the GBNF level, so `[7]` is unsamplable — an XGrammar assertion at decode time, not a post-hoc check.

In [ ]:
backend = HFBackend(model="Qwen/Qwen2.5-0.5B-Instruct", device="cpu")
cf = Citeformer(backend=backend, citation_policy=Policy.REQUIRED, style="apa-7")

result = cf.generate(prompt=prompt, sources=sources, max_new_tokens=220, temperature=0.3)
# Small instruction-tuned models sometimes stack adjacent cite-groups like
# "[1] [2] [3] [1]" when forced to close a sentence. Dedupe is a cheap
# cosmetic pass that preserves structural validity.
result_text = deduplicate_adjacent_cites(result.text)
print(result_text)

### Structural guarantee check

The grammar invariant is: every `[N]` marker in `result.text` has `1 ≤ N ≤ len(sources)`. Let's verify.

In [ ]:
emitted_ids = sorted({c.source_id for c in result.citations})
print(f"Sources in scope: 1..{len(sources)}")
print(f"Cite ids emitted: {emitted_ids}")
assert all(1 <= cid <= len(sources) for cid in emitted_ids), "structural invariant violated"
print("✓ every emitted citation is in-range (structural guarantee held)")

## 4. Verify — do the citations actually support the sentences they attach to?

Existence is trivial here (the grammar enforces it). The harder question is **entailment** — when the model cites `[3]` for a sentence, does source 3's abstract actually entail that sentence? We run NLI (DeBERTa-v3-large-MNLI by default) over every (sentence, source) pair that was cited.

This takes a minute on CPU — the DeBERTa model is ~850 MB and each claim is a forward pass.

In [ ]:
report = result.verify(threshold=0.5)
print(f"Citations scored: {report.citations_checked}")
print(f"Support rate: {report.support_rate * 100:.1f}%")
for per in report.per_citation[:10]:
    marker = result.citations[per.citation_index]
    sentence_start = max(0, marker.span[0] - 120)
    excerpt = result_text[sentence_start : marker.span[1]].strip()
    verdict = "✓" if per.supported else "✗"
    print(f"  {verdict} [source {marker.source_id}]  score={per.entailment_score:.2f}  …{excerpt[-100:]!r}")

Support rate on a 0.5B model is typically 20–50% with abstract-premise scoring. That's not a failure of the structural guarantee — the grammar can only prevent *fabricated* citations; it can't ensure small models cite the *right* source for each claim. `verify()` is what catches the latter.

## 5. Render a deterministic bibliography

The model doesn't write the bibliography — our formatter does. APA-7 here; `style='ieee'` / `'mla-9'` / `'chicago-author-date'` / `'nature'` / `'vancouver'` are all bundled.

In [ ]:
for ref in result.references:
    print(f"[{ref.source_id}] {ref.rendered}")

## 6. Compare — same prompt, no grammar layer

Now the structural guarantee contrast. We call the same model with the same prompt, but no grammar masking. The model can emit any `[N]` it wants.

In [ ]:
# The raw transformers pipeline, no citeformer — i.e. no grammar constraint at
# decode time. We use the same tokenizer+model instance (no model reload) so
# the only difference vs. the constrained run is the grammar mask.
import re

inputs = backend.tokenizer(prompt, return_tensors="pt").to(backend.device)
import torch  # noqa: E402 — transparent reimport

with torch.no_grad():
    output_ids = backend.model.generate(
        **inputs,
        max_new_tokens=220,
        temperature=0.3,
        do_sample=True,
        pad_token_id=backend.tokenizer.eos_token_id,
    )
baseline_text = backend.tokenizer.decode(output_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
print(baseline_text)

baseline_ids = sorted(set(int(m.group(1)) for m in re.finditer(r"\[(\d+)\]", baseline_text)))
fabricated = [cid for cid in baseline_ids if cid < 1 or cid > len(sources)]
print(f"\nBaseline cite ids emitted: {baseline_ids}")
print(f"Fabricated (out of range): {fabricated}")

## 7. Takeaways

- **Every citeformer-emitted citation was in-range.** Not because the model got lucky — because `[7]` is token-impossible to sample under the grammar mask. Run this 1000 times with 1000 different seeds and you'll get the same invariant every time.
- **Baseline runs sometimes emit out-of-range ids.** Whether it happened in *your* run depends on the seed and the prompt. It's not a reliable failure; it's a reliable *possibility* — which is enough for the guarantee to matter in production.
- **NLI verification surfaces the residual risk.** Grammar catches fabrication; NLI catches miscitation. Both layers are needed for a real RAG deployment.
- **The bibliography is deterministic.** Change styles with a single argument; the model isn't involved.

Extending this notebook:

- Swap to a bigger model (`microsoft/Phi-3.5-mini-instruct`, `meta-llama/Llama-3.2-3B-Instruct`) — support rate climbs.
- Swap the NLI premise to full paper body (`NLIModel(chunk_premise=True)` plus `Source.from_pdf`) — see `benchmarks/README.md` Finding 3 for the tradeoffs.
- Use an API backend (`citeformer.backends.openai.OpenAIBackend` / `.anthropic.AnthropicBackend`) for frontier-model output with schema-level cite enforcement.